In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
!pip install ultralytics -q
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.makedirs('/root/.config/kaggle', exist_ok=True)
with open('/root/.config/kaggle/kaggle.json', 'w') as f:
    f.write('{"username":"obiditislam","key":"KGAT_1d0a435832317a0f6dc2ad8632d87f66"}')
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)

!pip install kaggle -q
!kaggle datasets download -d banuprasadb/visdrone-dataset
!unzip -q visdrone-dataset.zip -d visdrone
print("Done")
print(open("visdrone/VisDrone_Dataset/visdrone.yaml").read())

In [ ]:
yaml_content = """
path: /content/visdrone/VisDrone_Dataset
train: VisDrone2019-DET-train/images
val: VisDrone2019-DET-val/images

nc: 10
names:
  0: pedestrian
  1: people
  2: bicycle
  3: car
  4: van
  5: truck
  6: tricycle
  7: awning-tricycle
  8: bus
  9: motor
"""

with open('/content/visdrone/VisDrone_Dataset/visdrone_fixed.yaml', 'w') as f:
    f.write(yaml_content)

# Verify labels exist and are being found correctly
from pathlib import Path
train_imgs = list(Path("visdrone/VisDrone_Dataset/VisDrone2019-DET-train/images").glob("*.jpg"))
train_lbls = list(Path("visdrone/VisDrone_Dataset/VisDrone2019-DET-train/labels").glob("*.txt"))
print(f"Train images: {len(train_imgs)}")
print(f"Train labels: {len(train_lbls)}")

# Verify a sample label has car (class 3)
sample = train_lbls[0]
lines = sample.read_text().strip().split("\n")
classes_in_sample = set(int(l.split()[0]) for l in lines if l)
print(f"Sample label classes: {classes_in_sample}")

In [ ]:
import os
path = '/content/drive/MyDrive/antlings/visdrone_v2/weights/last.pt'
print(os.path.exists(path))
print(os.path.getsize(path) / 1e6, "MB")

In [11]:
import os
from ultralytics import YOLO

checkpoint = '/content/drive/MyDrive/antlings/visdrone_v2_640/weights/last.pt'

if os.path.exists(checkpoint):
    print(f"Checkpoint found: {os.path.getsize(checkpoint)/1e6:.1f} MB")
    print("Resuming from checkpoint...")
    model = YOLO(checkpoint)
    model.train(resume=True)
else:
    print("No checkpoint found. Starting fresh...")
    model = YOLO('yolov8s.pt')
    model.train(
        data='/content/visdrone/VisDrone_Dataset/visdrone_fixed.yaml',
        epochs=50,
        imgsz=640,
        batch=16,
        name='visdrone_v2_640',
        project='/content/drive/MyDrive/antlings',
        patience=10,
        workers=4,
        cos_lr=True,
        plots=True,
    )

Checkpoint found: 89.5 MB
Resuming from checkpoint...
Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/visdrone/VisDrone_Dataset/visdrone_fixed.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/antlings/visdrone_v2_640/weights/last.pt, momentum=0.937, mosai